# Cattle Re-ID — Etapa 3, Fase 0: re-ID no supervisado por clustering (DINOv2)

Circuito end-to-end: **encoder congelado → embeber el target sin etiquetas → clusterizar (descubrir identidades) → evaluar**. Las etiquetas se usan **solo para evaluar**, nunca para clusterizar.

- **Target (campo nuevo, sin etiquetas):** Zenodo Muzzle DB (`BeefCattle_Muzzle_Individualized.zip`, 268 vacas).
- **Encoders (Fase 0):** DINOv2 (genérico auto-supervisado) + ImageNet ResNet-50 (piso genérico). Ninguno necesita entrenar.
- **Salida:** `outputs/results/10_cluster_summary.json` + tabla (ARI, NMI, nº clusters hallados vs real, Rank-1, mAP).

> **Prerrequisito:** pushear al repo los archivos nuevos (`src/reid/encoders.py`, `src/reid/cluster.py`, `scripts/10_cluster_reid.py`, cambios en `config.py` y `reid_dataset.py`) en la rama que se clona abajo.

## 0. GPU

In [1]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Sin GPU: Entorno de ejecución -> Cambiar tipo -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

GPU 0: Tesla T4 (UUID: GPU-06656dd9-29a0-c76c-8bd6-3b2467d358f1)
GPU: Tesla T4


## 1. Montar Drive

`DRIVE_ROOT` debe apuntar a la carpeta que contiene los `.zip`. Si la carpeta es **compartida**, agregá un acceso directo a *Mi unidad* (clic derecho → Organizar → Añadir acceso directo) y ajustá la ruta.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
# Ajustá esta ruta a la carpeta con los .zip (CMPD300_Baseline.zip, BeefCattle_Muzzle_Individualized.zip, ...).
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}. Ajustá DRIVE_ROOT.'
print('DRIVE_ROOT:', DRIVE_ROOT)
print('contenido:', [p.name for p in DRIVE_ROOT.iterdir()])

Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive/cattle_reid
contenido: ['BeefCattle_Muzzle_Individualized.zip', 'outputs', 'dataset_caras_bovinos.zip', 'gradcam_test', 'CMPD300_Baseline.zip', 'Untitled0.ipynb']


## 2. Traer el código (clon de tu repo/rama)

In [3]:
import os, shutil
os.chdir('/content')
REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
BRANCH   = 'stage3-fase0'   # rama con los archivos de la Fase 0 (Etapa 3)
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

Cloning into '/content/tp-final-vision2-Pujol-Vitale'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (194/194), done.
remote: Total 277 (delta 148), reused 198 (delta 75), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 257.52 KiB | 13.55 MiB/s, done.
Resolving deltas: 100% (148/148), done.
/content/tp-final-vision2-Pujol-Vitale
c497ef5 (HEAD -> stage3-fase0, origin/stage3-fase0) Etapa 3 Fase 0: re-ID no supervisado por clustering (DINOv2)


## 3. Dependencias

Colab ya trae torch/torchvision/scikit-learn. Solo falta `transformers` (para DINOv2). `sklearn.cluster.HDBSCAN` requiere scikit-learn ≥ 1.3 (Colab lo cumple).

In [4]:
!pip -q install 'transformers==4.40.2'
import sklearn; print('sklearn', sklearn.__version__)
from sklearn.cluster import HDBSCAN  # falla si sklearn < 1.3
print('HDBSCAN disponible ✅')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
sklearn 1.6.1
HDBSCAN disponible ✅


## 4. Persistir outputs en Drive (symlink)

Así el `10_cluster_summary.json` y el cache de HuggingFace quedan en Drive.

In [5]:
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
for sub in ('checkpoints', 'logs', 'results'):
    drive_sub = DRIVE_OUTPUTS / sub; drive_sub.mkdir(parents=True, exist_ok=True)
    local_sub = Path(REPO_DIR) / 'outputs' / sub
    if local_sub.exists() and not local_sub.is_symlink():
        shutil.rmtree(local_sub)
    if not local_sub.exists():
        os.symlink(drive_sub, local_sub, target_is_directory=True)
    print(f'outputs/{sub} -> {drive_sub}')

outputs/checkpoints -> /content/drive/MyDrive/cattle_reid/outputs/checkpoints
outputs/logs -> /content/drive/MyDrive/cattle_reid/outputs/logs
outputs/results -> /content/drive/MyDrive/cattle_reid/outputs/results


## 5. Extraer el Zenodo Muzzle DB (target)

Dataset chico (~643 MB); se extrae completo. Se autodetecta la carpeta con las 268 subcarpetas `cattle_*`.

In [6]:
import zipfile
MUZZLE_ZIP = DRIVE_ROOT / 'BeefCattle_Muzzle_Individualized.zip'
assert MUZZLE_ZIP.is_file(), f'Falta {MUZZLE_ZIP}.'
MUZZLE_LOCAL = Path('/content/muzzle')
if not MUZZLE_LOCAL.exists():
    MUZZLE_LOCAL.mkdir(parents=True)
    !unzip -q "{MUZZLE_ZIP}" -d "{MUZZLE_LOCAL}"

def find_dataset_root(base: Path) -> Path:
    # carpeta que contiene subcarpetas de clase (cattle_*/ con imágenes)
    for p in [base, *[d for d in base.rglob('*') if d.is_dir()]]:
        subs = [d for d in p.iterdir() if d.is_dir()]
        if len(subs) >= 100 and any(d.name.lower().startswith('cattle') for d in subs):
            return p
    raise RuntimeError('No encuentro la carpeta con las clases cattle_*.')

TARGET_DIR = find_dataset_root(MUZZLE_LOCAL)
n_ids = len([d for d in TARGET_DIR.iterdir() if d.is_dir()])
print('TARGET_DIR:', TARGET_DIR, '| clases:', n_ids)
# muestra de nombres reales de archivo (para chequear el naming de sesión)
ej = next(d for d in TARGET_DIR.iterdir() if d.is_dir())
print('ejemplo', ej.name, '->', [f.name for f in list(ej.iterdir())[:8]])

TARGET_DIR: /content/muzzle/BeefCattle_Muzzle_Individualized | clases: 268
ejemplo cattle_5507 -> ['cattle_5507_DSCF0311.jpg', 'cattle_5507_DSCF0304.jpg', 'cattle_5507_DSCF0303.jpg', 'cattle_5507_DSCF0300.jpg', 'cattle_5507_DSCF0293.jpg', 'cattle_5507_DSCF0305.jpg', 'cattle_5507_DSCF0310.jpg', 'cattle_5507_DSCF0306.jpg']


## 6. Correr la Fase 0 (DINOv2 + ImageNet)

Primero mirá el bloque **SESSION DIAGNOSTICS** en el log: dice si el split por sesión es real para este dataset (si `session_split_is_meaningful=False`, los nombres de archivo no codifican sesiones y el control anti-ráfaga es débil).

In [7]:
!python scripts/10_cluster_reid.py \
    --target-dir "{TARGET_DIR}" \
    --encoders dinov2 imagenet \
    --batch-size 64

[16:12:31] INFO target=/content/muzzle/BeefCattle_Muzzle_Individualized | 268 ids | 4923 images
[16:12:31] INFO == SESSION DIAGNOSTICS ==
[16:12:31] INFO     n_ids: 268
[16:12:31] INFO     n_images: 4923
[16:12:31] INFO     n_sessions: 4923
[16:12:31] INFO     avg_sessions_per_id: 18.37
[16:12:31] INFO     avg_images_per_session: 1.0
[16:12:31] INFO     ids_with_multi_session: 268
[16:12:31] INFO     frac_ids_multi_session: 1.0
[16:12:31] INFO     max_images_in_one_session: 1
[16:12:31] INFO     session_split_is_meaningful: False
[16:12:31] WARNING Session split looks DEGENERATE (≈1 photo per session): filenames may not encode capture sessions. The anti-burst control is then weak — clustering/retrieval on this target may reflect photo similarity. Check the filename convention before trusting session-based numbers.
[16:12:31] INFO anti-burst: 4923/4923 imgs kept (one per session) | retrieval split: gallery=268 probe=4655
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_downl

## 7. (Opcional) Sumar nuestros encoders de hocico

Si ya tenés `cmpd300_source.pt` (Fase 5) o el ArcFace en `outputs/checkpoints/` (vía el symlink a Drive), agregalos al mismo pipeline. Identidades disjuntas del source.

In [8]:
CKPT = Path(REPO_DIR)/'outputs/checkpoints/cmpd300_source.pt'
if CKPT.is_file():
    !python scripts/10_cluster_reid.py \
        --target-dir "{TARGET_DIR}" \
        --encoders dinov2 imagenet resnet-ckpt \
        --ckpt "{CKPT}" --batch-size 64
else:
    print('No hay cmpd300_source.pt; salteando. (Corré la Fase 5 o copialo a Drive.)')

[16:23:15] INFO target=/content/muzzle/BeefCattle_Muzzle_Individualized | 268 ids | 4923 images
[16:23:15] INFO == SESSION DIAGNOSTICS ==
[16:23:15] INFO     n_ids: 268
[16:23:15] INFO     n_images: 4923
[16:23:15] INFO     n_sessions: 4923
[16:23:15] INFO     avg_sessions_per_id: 18.37
[16:23:15] INFO     avg_images_per_session: 1.0
[16:23:15] INFO     ids_with_multi_session: 268
[16:23:15] INFO     frac_ids_multi_session: 1.0
[16:23:15] INFO     max_images_in_one_session: 1
[16:23:15] INFO     session_split_is_meaningful: False
[16:23:15] WARNING Session split looks DEGENERATE (≈1 photo per session): filenames may not encode capture sessions. The anti-burst control is then weak — clustering/retrieval on this target may reflect photo similarity. Check the filename convention before trusting session-based numbers.
[16:23:15] INFO anti-burst: 4923/4923 imgs kept (one per session) | retrieval split: gallery=268 probe=4655
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_downl

## 8. Resultados

In [9]:
import json
p = Path('outputs/results/10_cluster_summary.json')
res = json.loads(p.read_text())
print('session_diagnostics:', json.dumps(res['session_diagnostics'], indent=2))
for e in res['encoders']:
    if 'error' in e:
        print(e['encoder'], 'ERROR', e['error']); continue
    cd = e['clustering_session_dedup']
    best = max(cd['dbscan_grid'], key=lambda r: r['ARI'])
    hdb = cd['hdbscan'] or {}
    r = e['retrieval_session_single_shot'] or {}
    print(f"{e['encoder']:20} DBSCAN ARI={best['ARI']} (eps={best['eps']}, "
          f"{best['n_clusters_found']}/{cd['n_clusters_true']} clust) "
          f"HDBSCAN ARI={hdb.get('ARI')} Rank-1={r.get('rank1')}")

session_diagnostics: {
  "n_ids": 268,
  "n_images": 4923,
  "n_sessions": 4923,
  "avg_sessions_per_id": 18.37,
  "avg_images_per_session": 1.0,
  "ids_with_multi_session": 268,
  "frac_ids_multi_session": 1.0,
  "max_images_in_one_session": 1,
  "session_split_is_meaningful": false
}
dinov2-base          DBSCAN ARI=0.0123 (eps=0.1, 152/268 clust) HDBSCAN ARI=0.4027 Rank-1=0.7969924812030075
imagenet_resnet50    DBSCAN ARI=0.0017 (eps=0.1, 35/268 clust) HDBSCAN ARI=0.73 Rank-1=0.8938775510204081
cmpd300_source.pt    DBSCAN ARI=0.0139 (eps=0.1, 94/268 clust) HDBSCAN ARI=0.745 Rank-1=0.8635875402792696


## Cómo leer esto

- **ARI/NMI** cerca de 1 = el encoder descubrió las identidades sin que le digan cuántas hay.
- **nº clusters hallados vs real (268):** ¿sobre- o sub-segmenta?
- **DBSCAN** = mejor del grid de eps (el grid completo está en el JSON; eps NO se elige mirando labels en producción). **HDBSCAN** = estimación automática. **k-means** conoce el k real → techo con trampa.
- **Tesis:** si DINOv2 (genérico) iguala o supera a nuestros encoders de hocico, la especialización no ayudó a descubrir identidades — y eso es un hallazgo, no un fracaso.